# 05 — Table commune

**Étape du pipeline** : *Data transformation → Data tables*. C'est la table que D3 chargera avant le *visual mapping*.

Entrées : `data/interim/airbnb.parquet`, `data/interim/hotels.parquet`, et pour les quartiers `data/external/quartiers_<ville>.geojson` et `data/external/correspondance_quartiers.csv` (notebook 03).
Sorties, dans `data/processed/` : `offres.csv` (table complète) et un `offres_<ville>.csv` par ville, pour la consigne « changer de dataset ».

In [ ]:
import sys
from pathlib import Path

RACINE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "prep").is_dir())
sys.path.insert(0, str(RACINE))
RAW, INTERIM, PROCESSED = (RACINE / "data" / d for d in ("raw", "interim", "processed"))

import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

import geopandas as gpd
from src.prep import fusion, quartiers, schema
from src.prep.common import PERCENTILES_ABERRANTS

## 1. Fusion

In [ ]:
sources = [schema.conformer(pd.read_parquet(INTERIM / f)) for f in ("airbnb.parquet", "hotels.parquet")]
toutes = fusion.fusionner(*sources)
print(toutes.shape)
toutes.groupby(["ville", "source"], observed=True).size().unstack()

## 1 bis. Positions des hôtels

Le géocodage (notebook 04) a produit `data/external/hotels_geocodage.csv`. On ne garde que les points `fiable = true`, c'est-à-dire ceux qui passent les contrôles d'emprise, de distance au centre et de quartier. Les autres hôtels restent sans coordonnées.

La jointure passe par le lien Booking : `data/interim/hotels_liens.parquet` associe chaque `id` à son lien.

In [ ]:
EXTERNAL = RACINE / "data" / "external"
geocodage = pd.read_csv(EXTERNAL / "hotels_geocodage.csv")
liens = pd.read_parquet(INTERIM / "hotels_liens.parquet")
toutes = fusion.ajouter_positions_hotels(toutes, liens, geocodage)
h = toutes[toutes.source == "Hôtel"]
h.groupby("ville", observed=True)["lat"].agg(geolocalises="count", total="size").assign(taux=lambda t: (t.geolocalises / t.total).round(3))

## 1 ter. Rattachement aux quartiers

Les deux sources reçoivent un `quartier` pris dans le **même référentiel** : les contours du notebook 03. Ainsi, une carte peut les comparer zone par zone.

- Une ligne qui a des coordonnées est rattachée par **jointure spatiale**. C'est le cas de tous les Airbnb et des hôtels géocodés de façon fiable.
- Une ligne sans coordonnées garde le nom de quartier Booking, traduit par `correspondance_quartiers.csv`.

In [ ]:
contours = pd.concat([gpd.read_file(p) for p in sorted(EXTERNAL.glob("quartiers_*.geojson"))], ignore_index=True)
correspondance = quartiers.charger_correspondance(EXTERNAL / "correspondance_quartiers.csv")
toutes = fusion.ajouter_quartiers(toutes, contours, correspondance)
toutes.groupby(["ville", "source"], observed=True)["quartier"].apply(lambda s: s.notna().mean()).unstack().round(3)

# Origine du quartier des hôtels : position (jointure spatiale) ou nom Booking
h = toutes[toutes.source == "Hôtel"]
pd.crosstab(h.ville, h.lat.notna().map({True: "par position", False: "par nom Booking"}))

## 2. Prix aberrants

On écarte les prix hors des percentiles 1 %–99 %, calculés **par ville et par source**.

In [ ]:
offres, ecartees = fusion.retirer_aberrants(toutes)
fusion.bilan_effectifs(toutes, offres)

In [ ]:
ecartees.groupby(["ville", "source"], observed=True)["prix_nuit_eur"].agg(["min", "max"]).round(0)

## 3. Contrôles qualité

### Valeurs manquantes par source (%)

In [ ]:
fusion.valeurs_manquantes(offres)

Les colonnes vides d'une source sont **attendues** : il n'y a pas d'étoiles côté Airbnb, ni de coordonnées côté hôtels, etc. Les colonnes obligatoires sont contrôlées par `schema.valider`.

### Prix et notes par ville et par source

In [ ]:
offres.groupby(["ville", "source"], observed=True).agg(
    n=("id", "size"),
    prix_median=("prix_nuit_eur", "median"),
    prix_q1=("prix_nuit_eur", lambda s: s.quantile(.25)),
    prix_q3=("prix_nuit_eur", lambda s: s.quantile(.75)),
    note_mediane=("note_100", "median"),
    metro=("metro_proche", "mean"),
).round(2)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
for ax, (v, g) in zip(axes, offres.groupby("ville", observed=True)):
    ax.boxplot([g.loc[g.source == s, "prix_nuit_eur"] for s in ("Airbnb", "Hôtel")], showfliers=False)
    ax.set_xticks([1, 2], ["Airbnb", "Hôtel"]); ax.set_title(v)
axes[0].set_ylabel("€ / nuit"); plt.tight_layout()

**Notes**

- Les notes Airbnb sont très concentrées vers le haut (médiane autour de 95/100), davantage que les notes Booking. Cette différence vient des systèmes de notation des deux plateformes, pas forcément de la qualité. C'est une limite à signaler dans le rapport.
- Les prix des hôtels de Rome restent hors d'échelle. Ils sont gardés, avec un avertissement porté par `prix_comparable = False` (notebook 02, D7).

### Cohérence des distances au centre

In [ ]:
offres.groupby(["ville", "source"], observed=True)["dist_centre_km"].describe(percentiles=[.5]).round(1)

In [ ]:
pd.crosstab([offres.ville, offres.source], offres.tranche_dist, normalize="index").round(2)

Les deux sources ont des distributions de distance comparables : c'est ce qui permet de les comparer par tranche.

Attention : chaque dataset définit le « centre » à sa façon, et aucun des deux ne le documente.

### Couverture des quartiers

In [ ]:
couv = fusion.couverture_quartiers(offres, contours)
couv["comparable"] = (couv["Airbnb"] >= 30) & (couv["Hôtel"] >= 10)
couv.groupby(level="ville").agg(zones=("Airbnb", "size"),
                                 avec_airbnb=("Airbnb", lambda s: (s > 0).sum()),
                                 avec_hotels=("Hôtel", lambda s: (s > 0).sum()),
                                 comparables=("comparable", "sum"))

Une zone est dite **comparable** si elle compte au moins 30 Airbnb et 10 hôtels. Ce seuil est indicatif : l'interface D3 peut griser les autres zones.

- **Paris et Berlin** : presque toutes les zones sont comparables.
- **Londres** : le dataset Airbnb ne couvre que les boroughs centraux. Les boroughs périphériques sont vides côté Airbnb.
- **Rome** : les trois quarts des hôtels sont dans le Municipio I (centre historique). Seule cette zone est comparable (voir D7 et D16).

## 4. Export pour D3

Les contours simplifiés pour D3 (`data/processed/quartiers_<ville>.geojson`) sont écrits par le notebook 03.

In [ ]:
chemins = fusion.exporter(offres, PROCESSED)
for p in chemins:
    print(f"{p.relative_to(RACINE)}  {p.stat().st_size / 1e6:.2f} Mo")

In [ ]:
# Relecture : ce que D3 verra avec d3.csv(..., d3.autoType)
pd.read_csv(PROCESSED / "offres_paris.csv").sample(5, random_state=0)